# 02. Model architecture and weight transfer

**Goal:** understand which weights are copied from the pretrained checkpoint and which are randomly initialized when you fine-tune on a new task.

In [ ]:
# bootstrap: make course_utils + scripts importable from any working directory,
# use the inline backend so figures render, and regenerate the phantom if missing.
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO = _find_repo_root(Path.cwd())
for _p in (str(REPO), str(REPO / "scripts")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

DATA = REPO / "assets" / "data" / "Dataset999_Phantom"
PRE = REPO / "assets" / "precomputed"

if not (DATA / "imagesTr" / "PHANTOM_001_0000.nii.gz").exists():
    import generate_phantom
    generate_phantom.generate(REPO / "assets" / "data")

print("repo root:", REPO.name)


## What is in a trained network

A trained PyTorch network is stored as a *state dict*: a dictionary that maps each layer's parameter name (a string such as `encoder.stages.0.convs.0.conv.weight`) to its weight tensor.

Fine-tuning splits these parameters into two groups:
- **transfer**: copy the tensor from the pretrained checkpoint (the encoder and the decoder body, which carry general anatomy);
- **re-initialize**: start from random weights (the segmentation outputs, which depend on the new task's label set).

nnU-Net decides this **by parameter name**: it loads every tensor whose name does not contain `.seg_layers.`, and randomly initializes the rest. Because nnU-Net uses **deep supervision**, segmentation outputs exist at several decoder depths, so there is not a single final layer; all of them are re-initialized.

## A small concrete network

A miniature module with the same structure: a shared body plus one segmentation head per decoder scale (deep supervision). We inspect its real `state_dict` keys.

In [ ]:
import torch
import torch.nn as nn

class TinyDeepSupUNet(nn.Module):
    def __init__(self, ch=4, n_classes=2, n_scales=3):
        super().__init__()
        self.encoder = nn.ModuleList([nn.Conv3d(1 if i == 0 else ch, ch, 3, padding=1) for i in range(n_scales)])
        self.decoder = nn.ModuleList([nn.Conv3d(ch, ch, 3, padding=1) for _ in range(n_scales)])
        self.seg_layers = nn.ModuleList([nn.Conv3d(ch, n_classes, 1) for _ in range(n_scales)])

net = TinyDeepSupUNet()
print('this toy net has', len(net.state_dict()), 'parameter tensors')

## Apply nnU-Net's rule: load everything except `.seg_layers.`

For each parameter we show its shape, which part of the network it belongs to, and whether fine-tuning transfers it or re-initializes it.

In [ ]:
SKIP = '.seg_layers.'
print(f"{'parameter name':40s} {'shape':16s} {'part':10s} action")
print('-' * 80)
for name, tensor in net.state_dict().items():
    part = 'encoder' if name.startswith('encoder') else ('seg head' if 'seg_layers' in name else 'decoder')
    action = 're-init (random)' if SKIP in ('.' + name) else 'transfer (load)'
    print(f'{name:40s} {str(tuple(tensor.shape)):16s} {part:10s} {action}')

## Why re-initialize the heads?

The pretraining heads predict the pretraining datasets' classes, which mean nothing for your lesion label. You drop them, attach fresh heads, and let fine-tuning learn the new mapping on top of the transferred encoder and decoder body.

Caveat: if your class count matches the pretraining, nnU-Net may load the old heads instead. Re-initialization is not automatic; it follows from skipping the `.seg_layers.` keys.

Real fine-tuning command (read-only):
```bash
nnUNetv2_train 999 3d_fullres 0 -tr nnUNetTrainer_warmup1e3 \
    -pretrained_weights /path/to/checkpoint_final.pth
```

## Recap
1. A network is a dictionary of named weight tensors.
2. Fine-tuning transfers the encoder and decoder body and re-initializes the segmentation outputs.
3. The choice is made by name (`.seg_layers.`), and deep supervision means there are several seg layers.